In [1]:
!pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 103.8 MB/s eta 0:00:00
  Attempting uninstall: autograd
    Found existing installation: autograd 1.9.1
    Uninstalling autograd-1.9.1:
      Successfully uninstalled autograd-1.9.1


In [2]:
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qml
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ==========================================
# 1. VERİ HAZIRLIĞI VE SIZINTISIZ BÖLME
# ==========================================
print("--- [KOD 5] ANN VE QNN (5-FOLD CV) DENEYLERİ BAŞLATILIYOR ---")
df = pd.read_csv("creditcard.csv")

fraud = df[df["Class"] == 1]
n_fraud = len(fraud)
normal = df[df["Class"] == 0].sample(n=n_fraud, random_state=RANDOM_SEED)
df_balanced = (
    pd.concat([fraud, normal])
    .sample(frac=1, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)

X_raw = df_balanced.drop(["Class"], axis=1).values
y_raw = df_balanced["Class"].values
input_dim = X_raw.shape[1]

print(f"Toplam Dengelenmiş Örneklem: {len(df_balanced)} (Fraud: {n_fraud}, Normal: {n_fraud})")
print(f"Girdi Boyutu: {input_dim}\n")

# ==========================================
# 2. MODEL TANIMLARI
# ==========================================
def create_hybrid_qnn_class(n_qubits, n_layers=2):
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def qnode(inputs, weights):
        qml.templates.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
        qml.templates.StronglyEntanglingLayers(weights, wires=range(n_qubits))
        return [qml.expval(qml.PauliZ(wires=i)) for i in range(n_qubits)]

    weight_shapes = {"weights": (n_layers, n_qubits, 3)}

    class DynamicHybridQNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.fc1 = nn.Linear(input_dim, 16)
            self.dropout = nn.Dropout(0.3)
            self.fc_bottleneck = nn.Linear(16, n_qubits)
            self.qlayer = qml.qnn.TorchLayer(qnode, weight_shapes)
            self.fc_out = nn.Linear(n_qubits, 1)
            self.sigmoid = nn.Sigmoid()

        def forward(self, x):
            x = torch.relu(self.fc1(x))
            x = self.dropout(x)
            x = torch.relu(self.fc_bottleneck(x))
            x = torch.tanh(x) * np.pi
            x = self.qlayer(x)
            x = self.sigmoid(self.fc_out(x))
            return x

    return DynamicHybridQNN

class PureClassicalANN(nn.Module):
    def __init__(self, bottleneck_dim=2):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 16)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(16, bottleneck_dim)
        self.fc3 = nn.Linear(bottleneck_dim, bottleneck_dim)
        self.fc_out = nn.Linear(bottleneck_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = self.sigmoid(self.fc_out(x))
        return x

# ==========================================
# 3. 5-FOLD ÇAPRAZ DOĞRULAMA MOTORU
# ==========================================
def evaluate_model_kfold(model_builder_fn, model_name, epochs=30, lr=0.002, batch_size=32):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

    fold_acc = []
    fold_f1_bin = []
    fold_f1_macro = []
    fold_auc = []

    print(f"--- [{model_name}] 5-Fold Çapraz Doğrulama Başlatılıyor ({epochs} Epoch) ---")
    start_total_t = time.time()

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_raw, y_raw), 1):
        # 1. Fold bazlı Data Leakage önleme: Scaler YALNIZCA train katında fit edilir
        X_tr_raw, X_va_raw = X_raw[train_idx], X_raw[val_idx]
        y_tr_np, y_va_np = y_raw[train_idx], y_raw[val_idx]

        scaler = MinMaxScaler(feature_range=(-1, 1))
        X_tr_scaled = scaler.fit_transform(X_tr_raw)
        X_va_scaled = scaler.transform(X_va_raw)

        # DataLoader hazırlığı
        train_ds = TensorDataset(torch.tensor(X_tr_scaled, dtype=torch.float32),
                                 torch.tensor(y_tr_np, dtype=torch.float32))
        val_ds = TensorDataset(torch.tensor(X_va_scaled, dtype=torch.float32),
                               torch.tensor(y_va_np, dtype=torch.float32))

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

        # Her fold için sıfır model örneği
        model = model_builder_fn()
        criterion = nn.BCELoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        # Eğitim
        for epoch in range(epochs):
            model.train()
            for bx, by in train_loader:
                optimizer.zero_grad()
                out = model(bx)
                loss = criterion(out, by.unsqueeze(1))
                loss.backward()
                optimizer.step()

        # Değerlendirme
        model.eval()
        y_true, y_pred, y_probs = [], [], []
        with torch.no_grad():
            for bx, by in val_loader:
                out = model(bx)
                preds = (out >= 0.5).float()
                y_true.extend(by.numpy())
                y_pred.extend(preds.numpy().flatten())
                y_probs.extend(out.numpy().flatten())

        acc = accuracy_score(y_true, y_pred)
        f1_b = f1_score(y_true, y_pred, average="binary", zero_division=0)
        f1_m = f1_score(y_true, y_pred, average="macro", zero_division=0)
        auc = roc_auc_score(y_true, y_probs)

        fold_acc.append(acc)
        fold_f1_bin.append(f1_b)
        fold_f1_macro.append(f1_m)
        fold_auc.append(auc)
        print(f"   Fold {fold}/5: Acc: %{acc*100:.2f} | F1: %{f1_b*100:.2f} | AUC: {auc:.3f}")

    total_elapsed = time.time() - start_total_t
    print(f"[{model_name}] Tamamlandı (Toplam Süre: {total_elapsed:.1f} sn)\n")

    return {
        "acc_mean": np.mean(fold_acc) * 100,
        "acc_std": np.std(fold_acc) * 100,
        "f1_bin_mean": np.mean(fold_f1_bin) * 100,
        "f1_bin_std": np.std(fold_f1_bin) * 100,
        "f1_macro_mean": np.mean(fold_f1_macro) * 100,
        "f1_macro_std": np.std(fold_f1_macro) * 100,
        "auc_mean": np.mean(fold_auc),
        "auc_std": np.std(fold_auc),
        "total_time": total_elapsed
    }

# ==========================================
# 4. MODELLERİN 5-FOLD CV ÇALIŞTIRILMASI
# ==========================================
cv_summary = {}

# Hibrit QNN Modelleri (30 Epoch)
QNN2_Builder = create_hybrid_qnn_class(n_qubits=2, n_layers=2)
cv_summary["Hibrit QNN (2 Kübit, 30 Ep)"] = evaluate_model_kfold(QNN2_Builder, "Hibrit QNN (2 Kübit)", epochs=30)

QNN4_Builder = create_hybrid_qnn_class(n_qubits=4, n_layers=2)
cv_summary["Hibrit QNN (4 Kübit, 30 Ep)"] = evaluate_model_kfold(QNN4_Builder, "Hibrit QNN (4 Kübit)", epochs=30)

QNN8_Builder = create_hybrid_qnn_class(n_qubits=8, n_layers=2)
cv_summary["Hibrit QNN (8 Kübit, 30 Ep)"] = evaluate_model_kfold(QNN8_Builder, "Hibrit QNN (8 Kübit)", epochs=30)

# Klasik ANN Modelleri
ANN_Builder = lambda: PureClassicalANN(bottleneck_dim=2)
cv_summary["Klasik ANN (0 Kübit, 30 Ep)"] = evaluate_model_kfold(ANN_Builder, "Klasik ANN (30 Epoch)", epochs=30)
cv_summary["Klasik ANN (0 Kübit, 100 Ep)"] = evaluate_model_kfold(ANN_Builder, "Klasik ANN (100 Epoch)", epochs=100)
cv_summary["Klasik ANN (0 Kübit, 200 Ep)"] = evaluate_model_kfold(ANN_Builder, "Klasik ANN (200 Epoch)", epochs=200)

# ==========================================
# 5. HAKEM #2 YORUM #4 İÇİN SONUÇ TABLOSU
# ==========================================
print("\n" + "=" * 110)
print(f"{'Model Mimarisi':<32} | {'Doğruluk (Mean ± SD)':<22} | {'F1 Skoru (Mean ± SD)':<22} | {'ROC-AUC':<16} | {'Süre'}")
print("=" * 110)
for name, stats in cv_summary.items():
    acc_str = f"%{stats['acc_mean']:.2f} ± {stats['acc_std']:.2f}"
    f1_str = f"%{stats['f1_bin_mean']:.2f} ± {stats['f1_bin_std']:.2f}"
    auc_str = f"{stats['auc_mean']:.3f} ± {stats['auc_std']:.3f}"
    print(f"{name:<32} | {acc_str:<22} | {f1_str:<22} | {auc_str:<16} | {stats['total_time']:.1f} sn")
print("=" * 110)

--- [KOD 5] ANN VE QNN (5-FOLD CV) DENEYLERİ BAŞLATILIYOR ---
Toplam Dengelenmiş Örneklem: 984 (Fraud: 492, Normal: 492)
Girdi Boyutu: 30

--- [Hibrit QNN (2 Kübit)] 5-Fold Çapraz Doğrulama Başlatılıyor (30 Epoch) ---
   Fold 1/5: Acc: %92.39 | F1: %91.71 | AUC: 0.957
   Fold 2/5: Acc: %94.42 | F1: %94.36 | AUC: 0.981
   Fold 3/5: Acc: %90.86 | F1: %90.22 | AUC: 0.969
   Fold 4/5: Acc: %90.86 | F1: %90.43 | AUC: 0.943
   Fold 5/5: Acc: %93.37 | F1: %93.40 | AUC: 0.981
[Hibrit QNN (2 Kübit)] Tamamlandı (Toplam Süre: 66.1 sn)

--- [Hibrit QNN (4 Kübit)] 5-Fold Çapraz Doğrulama Başlatılıyor (30 Epoch) ---
   Fold 1/5: Acc: %94.92 | F1: %94.74 | AUC: 0.987
   Fold 2/5: Acc: %94.92 | F1: %94.85 | AUC: 0.988
   Fold 3/5: Acc: %91.37 | F1: %90.81 | AUC: 0.964
   Fold 4/5: Acc: %90.86 | F1: %90.62 | AUC: 0.950
   Fold 5/5: Acc: %94.90 | F1: %94.79 | AUC: 0.979
[Hibrit QNN (4 Kübit)] Tamamlandı (Toplam Süre: 100.2 sn)

--- [Hibrit QNN (8 Kübit)] 5-Fold Çapraz Doğrulama Başlatılıyor (30 Epoch) -